#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [2]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive


#`Setup And Update ComfyUI`



In [ ]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

# `Models Download`

## Download MODELS

In [3]:
%cd {WORKSPACE}

# 安装 Aria2（如果还没装）
!apt-get -y install -qq aria2

# 读取 Civitai API Key（需提前在 Secrets 中设置）
from google.colab import userdata
CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
print("Loaded API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found")

# 定义下载函数（如果笔记本里已有则跳过）
def downloadModel(url, filename=None):
    if 'huggingface.co' in url:
        if filename is None:
            filename = url.split('/')[-1]
            filename = filename.removesuffix('?download=true')
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url} -o {filename}
    else:
        # civitai
        if filename:
            !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN} -o {filename}
        else:
            !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN}

# ========== 下载主模型（GGUF）到 checkpoints ==========
%cd ./models/checkpoints
print("⬇️ 下载 WAI Illustrious 基础模型...")
downloadModel('https://huggingface.co/kekusprod/WAI-NSFW-illustrious-SDXL-v110-GGUF/resolve/main/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf')

# ========== 下载文本编码器到 text_encoders ==========
%cd ../text_encoders
print("⬇️ 下载 CLIP G...")
downloadModel('https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_clip_g_fp8_e4m3fn.safetensors')
print("⬇️ 下载 CLIP L...")
downloadModel('https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_clip_l_fp8_e4m3fn.safetensors')

# ========== 下载 VAE 到 vae ==========
%cd ../vae
print("⬇️ 下载 VAE...")
downloadModel('https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_v110_vae_fp8_e4m3fn.safetensors')

# ========== 下载 LoRA 到 loras ==========
%cd ../loras
print("⬇️ 下载越山弱衰 LoRA...")
downloadModel('https://civitai.com/api/download/models/1590157?fileId=1490031', 'etuzan_jakusui_illustrious.safetensors')

# 回到工作区
%cd {WORKSPACE}
print("\n✅ 所有文件下载完成！")

[Errno 2] No such file or directory: '{WORKSPACE}'
/content
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not 

## OTHERS

In [ ]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

## LIST MODELS

In [ ]:
!ls -al ./models/checkpoints/

total 6775444
drwxr-xr-x  2 root root       4096 Aug 30 23:40 .
drwxr-xr-x 22 root root       4096 Aug 30 23:39 ..
-rw-r--r--  1 root root 6938040682 Aug 30 23:40 cyberrealisticPony_v127Alt.safetensors
-rw-r--r--  1 root root          0 Aug 30 23:39 put_checkpoints_here


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!python main.py --dont-print-server

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [11]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-07-15 09:27:27--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.7.1/cloudflared-linux-amd64.deb [following]
--2026-07-15 09:27:27--  https://github.com/cloudflare/cloudflared/releases/download/2026.7.1/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/d5091923-fdcd-41e7-a4f9-1c74ed2c2255?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-15T10%3A19%3A48Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

## CF Tunnel

In [17]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

python3: can't open file '/content/ComfyUI/main.py': [Errno 2] No such file or directory


## localtunnel

In [13]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.1
npm notice To update run: npm install -g npm@12.0.1
npm notice
⠏/content/ComfyUI
python3: can't open file '/content/ComfyUI/main.py': [Errno 2] No such file or directory


In [14]:
# 移动已下载的文件到正确位置
!mkdir -p /content/ComfyUI/models/checkpoints
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae
!mkdir -p /content/ComfyUI/models/loras

# 移动文件
!mv /content/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf /content/ComfyUI/models/checkpoints/
!mv /content/illustrious_clip_g_fp8_e4m3fn.safetensors /content/ComfyUI/models/text_encoders/
!mv /content/illustrious_clip_l_fp8_e4m3fn.safetensors /content/ComfyUI/models/text_encoders/
!mv /content/illustrious_v110_vae_fp8_e4m3fn.safetensors /content/ComfyUI/models/vae/

print("✅ 文件已移动到正确位置")

mv: cannot stat '/content/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf': No such file or directory
mv: cannot stat '/content/illustrious_clip_g_fp8_e4m3fn.safetensors': No such file or directory
mv: cannot stat '/content/illustrious_clip_l_fp8_e4m3fn.safetensors': No such file or directory
mv: cannot stat '/content/illustrious_v110_vae_fp8_e4m3fn.safetensors': No such file or directory
✅ 文件已移动到正确位置


In [15]:
!cd /content/ComfyUI/models/loras && wget --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36" -O etuzan_jakusui_illustrious.safetensors "https://civitai.com/api/download/models/1590157?fileId=1490031&token=70da68393ee28b3ffa78e834976edf77"

--2026-07-15 09:27:31--  https://civitai.com/api/download/models/1590157?fileId=1490031&token=70da68393ee28b3ffa78e834976edf77
Resolving civitai.com (civitai.com)... 172.66.152.186, 104.20.38.219, 2606:4700:10::6814:26db, ...
Connecting to civitai.com (civitai.com)|172.66.152.186|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: https://b2.civitai.com/file/civitai-modelfiles/model/20708/illEtuzanJakusui.UZuA.safetensors?Authorization=3_20260715092731_8707cba513cccfb55dbdfc79_9876a3ba272d6fac559be81b6e6a469ba9b6c99e_004_20260715102731_0045_dnld&b2ContentDisposition=attachment%3B+filename%3D%22ill_etuzan_jakusui_styleV1.safetensors%22 [following]
--2026-07-15 09:27:31--  https://b2.civitai.com/file/civitai-modelfiles/model/20708/illEtuzanJakusui.UZuA.safetensors?Authorization=3_20260715092731_8707cba513cccfb55dbdfc79_9876a3ba272d6fac559be81b6e6a469ba9b6c99e_004_20260715102731_0045_dnld&b2ContentDisposition=attachment%3B+filename%3D%22ill_etuzan_j

In [16]:
# 确保目录存在
!mkdir -p /content/ComfyUI/models/checkpoints
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae

# 1. 下载主模型（GGUF）到 checkpoints
print("⬇️ 下载主模型...")
!wget -c --header="User-Agent: Mozilla/5.0" -O /content/ComfyUI/models/checkpoints/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf "https://huggingface.co/kekusprod/WAI-NSFW-illustrious-SDXL-v110-GGUF/resolve/main/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf"

# 2. 下载 CLIP G
print("⬇️ 下载 CLIP G...")
!wget -c --header="User-Agent: Mozilla/5.0" -O /content/ComfyUI/models/text_encoders/illustrious_clip_g_fp8_e4m3fn.safetensors "https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_clip_g_fp8_e4m3fn.safetensors"

# 3. 下载 CLIP L
print("⬇️ 下载 CLIP L...")
!wget -c --header="User-Agent: Mozilla/5.0" -O /content/ComfyUI/models/text_encoders/illustrious_clip_l_fp8_e4m3fn.safetensors "https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_clip_l_fp8_e4m3fn.safetensors"

# 4. 下载 VAE
print("⬇️ 下载 VAE...")
!wget -c --header="User-Agent: Mozilla/5.0" -O /content/ComfyUI/models/vae/illustrious_v110_vae_fp8_e4m3fn.safetensors "https://huggingface.co/calcuis/illustrious/resolve/main/illustrious_v110_vae_fp8_e4m3fn.safetensors"

print("✅ 所有文件下载完成！")

⬇️ 下载主模型...
--2026-07-15 09:27:35--  https://huggingface.co/kekusprod/WAI-NSFW-illustrious-SDXL-v110-GGUF/resolve/main/WAI-NSFW-illustrious-SDXL-v110-Q4_K_S.gguf
Resolving huggingface.co (huggingface.co)... 3.166.152.44, 3.166.152.105, 3.166.152.65, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.44|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/67bbaa126b4c76145df04ea4/0dc730e3ee79d3c89ec74faf02b04b02e0653b3f4b236ed6b6856211ee9f1a3d?Expires=1784111255&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY3YmJhYTEyNmI0Yzc2MTQ1ZGYwNGVhNC8wZGM3MzBlM2VlNzlkM2M4OWVjNzRmYWYwMmIwNGIwMmUwNjUzYjNmNGIyMzZlZDZiNjg1NjIxMWVlOWYxYTNkKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDExMTI1NX19fV19&Signature=MEUCIQCETQXHXMscCREYBMHVGWqGSqanC6ZS3xXxZXuAJ3-xiQIgAJgxplBR15cmobraNKxfcuUif%7E44uNMFNfO6NawpScg_&Key-Pair-Id=K3EPXBYC3CKDRZ&response-

In [19]:
!ls -la /content/ComfyUI/

total 12
drwxr-xr-x 3 root root 4096 Jul 15 09:20 .
drwxr-xr-x 1 root root 4096 Jul 15 09:27 ..
drwxr-xr-x 6 root root 4096 Jul 15 09:20 models


In [20]:
%cd /content/ComfyUI
import subprocess, threading, time, socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI loaded, launching cloudflared...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
!python main.py --dont-print-server

/content/ComfyUI
python3: can't open file '/content/ComfyUI/main.py': [Errno 2] No such file or directory


In [22]:
# 1. 强行回到 /content
%cd /content

# 2. 如果 ComfyUI 存在但残缺，删除它
!rm -rf /content/ComfyUI

# 3. 重新克隆 ComfyUI（不指定子目录，让它在当前目录创建）
!git clone https://github.com/comfyanonymous/ComfyUI.git

# 4. 进入 ComfyUI
%cd /content/ComfyUI

# 5. 安装依赖
!pip install -r requirements.txt

# 6. 验证 main.py
!ls -la main.py

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 43231, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 43231 (delta 15), reused 6 (delta 5), pack-reused 43198 (from 2)
Receiving objects: 100% (43231/43231), 89.92 MiB | 23.85 MiB/s, done.
Resolving deltas: 100% (29311/29311), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 132.1 MB/s

In [23]:
%cd /content/ComfyUI

import subprocess, threading, time, socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI loaded, launching cloudflared...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
!python main.py --dont-print-server


/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_